# Persistent Memory for the Claude Agent SDK with Hindsight

[Hindsight](https://github.com/vectorize-io/hindsight) is an open-source (MIT) long-term memory engine for AI agents. This notebook shows how to give a [Claude Agent SDK](https://github.com/anthropics/claude-agent-sdk-python) agent memory that persists across sessions, using the [`hindsight-claude-agent-sdk`](https://pypi.org/project/hindsight-claude-agent-sdk/) package.

You get two integration patterns:

- **In-process MCP tools** — `hindsight_retain`, `hindsight_recall`, `hindsight_reflect`, which the agent calls explicitly.
- **Automatic memory hooks** — relevant memories are injected before each prompt (`UserPromptSubmit`) and the result is retained afterward (`Stop`), with no explicit tool calls.

## Prerequisites

- **Claude Code CLI** installed and authenticated — the Claude Agent SDK runs it as a subprocess (`npm install -g @anthropic-ai/claude-code && claude auth login`, or set `ANTHROPIC_API_KEY`).
- An LLM API key for Hindsight (OpenAI, Gemini, etc.).
- A running Hindsight instance — locally via Docker (below), or a free [Hindsight Cloud](https://hindsight.vectorize.io) account (no Docker needed).

## Start Hindsight locally

Before running this notebook, start Hindsight in a terminal:

```bash
export LLM_API_KEY="your-llm-api-key"

docker run --rm -it --pull always -p 8888:8888 -p 9999:9999 \
  -e HINDSIGHT_API_LLM_API_KEY=$LLM_API_KEY \
  -e HINDSIGHT_API_LLM_MODEL=gpt-4o-mini \
  -v $HOME/.hindsight-docker:/home/hindsight/.pg0 \
  ghcr.io/vectorize-io/hindsight:latest
```


## 1. Install dependencies

In [ ]:
!pip install -q hindsight-claude-agent-sdk nest-asyncio

## 2. Configure environment

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import getpass

# Anthropic API key (used by the Claude Agent SDK / Claude Code CLI)
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

# Hindsight connection (defaults to a local self-hosted instance).
# For Hindsight Cloud, set HINDSIGHT_API_URL and HINDSIGHT_API_KEY.
HINDSIGHT_API_URL = os.getenv("HINDSIGHT_API_URL", "http://localhost:8888")
HINDSIGHT_API_KEY = os.getenv("HINDSIGHT_API_KEY", None)  # optional, for Hindsight Cloud
BANK_ID = "claude-agent-demo"

print(f"Hindsight API: {HINDSIGHT_API_URL}")
print(f"Using API key: {'yes' if HINDSIGHT_API_KEY else 'no (self-hosted)'}")
print(f"Bank ID: {BANK_ID}")

## 3. Create a memory bank

In [ ]:
from hindsight_client import Hindsight

hindsight = Hindsight(base_url=HINDSIGHT_API_URL, api_key=HINDSIGHT_API_KEY)

# Create a dedicated bank for this demo (safe to re-run)
try:
    hindsight.create_bank(
        bank_id=BANK_ID,
        name="Claude Agent Demo",
        mission="Remember user preferences, decisions, and project context for a software development assistant.",
    )
    print(f"Bank '{BANK_ID}' created.")
except Exception:
    print(f"Bank '{BANK_ID}' already exists, continuing.")

## 4. Set up memory tools

Create an in-process MCP server exposing retain, recall, and reflect tools:

In [ ]:
from hindsight_claude_agent_sdk import create_hindsight_server

server = create_hindsight_server(
    bank_id=BANK_ID,
    hindsight_api_url=HINDSIGHT_API_URL,
    api_key=HINDSIGHT_API_KEY,
    tags=["source:claude-agent-sdk-demo"],
)

print("Hindsight MCP server created with tools: hindsight_retain, hindsight_recall, hindsight_reflect")

## 5. Run the agent with explicit memory tools

The agent decides when to store and retrieve memories:

In [ ]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions

async def run_agent(prompt: str, system: str = None):
    """Run a Claude agent with Hindsight memory tools."""
    options = ClaudeAgentOptions(
        mcp_servers={"hindsight": server},
        allowed_tools=["mcp__hindsight__*"],
        model="claude-sonnet-4-6",
        permission_mode="bypassPermissions",
    )
    if system:
        options.system_prompt = system

    result_text = None
    async for msg in query(prompt=prompt, options=options):
        if hasattr(msg, "result"):
            result_text = msg.result
    return result_text


# Store some preferences
result = asyncio.get_event_loop().run_until_complete(
    run_agent(
        "Store the following into memory using the retain tool:\n"
        "- I prefer Python with type hints and async/await patterns\n"
        "- My team uses pytest for testing with pytest-asyncio\n"
        "- We follow conventional commits (feat:, fix:, chore:)\n"
        "- Our API framework is FastAPI with Pydantic v2 models"
    )
)
print("Agent result:", result)

## 6. Recall memories in a new session

Simulate a fresh session — the agent has no conversation history, but can recall from memory:

In [ ]:
# New session — no prior context
result = asyncio.get_event_loop().run_until_complete(
    run_agent(
        "What testing framework does my team use? "
        "Search your memory first before answering.",
        system="You are a helpful coding assistant. Always check memory before answering questions about the user.",
    )
)
print("Agent result:", result)

## 7. Reflect for deeper synthesis

Use reflect when you need reasoned analysis across all stored memories:

In [ ]:
result = asyncio.get_event_loop().run_until_complete(
    run_agent(
        "Use the reflect tool to synthesize everything you know about my development stack and preferences.",
    )
)
print("Agent result:", result)

## 8. Add automatic memory hooks

Hooks inject memory automatically — no explicit tool calls needed:

In [ ]:
from hindsight_claude_agent_sdk import create_memory_hooks, MemoryHookConfig

hooks = create_memory_hooks(
    bank_id=BANK_ID,
    hindsight_api_url=HINDSIGHT_API_URL,
    api_key=HINDSIGHT_API_KEY,
    hook_config=MemoryHookConfig(
        auto_recall=True,       # inject relevant memories before each prompt
        auto_retain=True,       # save agent results after each session
        recall_max_results=5,   # limit injected memories
    ),
)

print("Memory hooks created: auto-recall on UserPromptSubmit, auto-retain on Stop")

## 9. Run the agent with hooks

Now the agent gets relevant memories injected as system context automatically:

In [ ]:
async def run_with_hooks(prompt: str):
    """Run a Claude agent with both tools and hooks."""
    result_text = None
    async for msg in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            mcp_servers={"hindsight": server},
            allowed_tools=["mcp__hindsight__*"],
            hooks=hooks,
            model="claude-sonnet-4-6",
            permission_mode="bypassPermissions",
            system_prompt="You are a helpful coding assistant.",
        ),
    ):
        if hasattr(msg, "result"):
            result_text = msg.result
    return result_text


# The agent receives past memories automatically — no tool call needed
result = asyncio.get_event_loop().run_until_complete(
    run_with_hooks("Write a sample pytest test for a FastAPI endpoint, using my team's preferred patterns.")
)
print("Agent result:", result)

The agent received your team's testing preferences via auto-recall before it even started working — and its result was auto-retained for future sessions.

## 10. Run again to see knowledge compound

Each session adds to the knowledge base. Run the agent again with a related prompt:

In [ ]:
result = asyncio.get_event_loop().run_until_complete(
    run_with_hooks("What commit message format should I use for this test file I just created?")
)
print("Agent result:", result)

The agent recalls the conventional-commits preference from earlier — even though it was stored in a completely different session.

## Cleanup

Delete the bank created during this notebook:

In [ ]:
hindsight.delete_bank(bank_id=BANK_ID)
print(f"Deleted bank '{BANK_ID}'.")